# 암종 분류 v3: 이진 행렬 + 파생변수 + 인프레임 핫스팟 + 동반변이 콤보 + XGB/LR 앙상블 (5-fold CV)

**v2 대비 변경점**
- 인프레임 변이(`EGFR E746_A750del` 등)를 핫스팟 토큰으로 추가 (`gene|inframe`, `gene|위치_inframe`)
- 가이드 기반 동반변이 콤보 피처 (`cb_*`), TMB 기준 5 / 30
- **EM 사전확률 보정 제거** (macroF1을 0.46 -> 0.21로 망가뜨림). 대신 COAD 하나만 약하게 키우는 보정을 시뮬레이션으로 검증 후 자동 적용(이득 없으면 미적용)
- 실행 순서 버그 수정: 예전 prior 셀이 CV보다 앞에 있어 `oof_ens`가 정의되기 전에 쓰였음 -> 앙상블 이후로 이동
- CV를 함수(`run_cv`)로 묶고, 콤보 ON/OFF 비교(ablation) 셀 추가

**패키지**: `pip install pandas numpy scikit-learn xgboost`


In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
warnings.filterwarnings("ignore")

CFG = dict(
    DATA_DIR=os.environ.get("DATA_DIR", "../../data"),
    SEED=42,
    N_FOLDS=int(os.environ.get("N_FOLDS", 5)),
    MIN_GENE_FREQ=5,        # train에서 변이 샘플이 이 값 미만인 유전자는 제외
    MIN_HOTSPOT_FREQ=5,     # gene|position 토큰 최소 빈도
    MIN_AA_FREQ=20,         # 아미노산 치환쌍 최소 빈도
    USE_AA_SUBS=True,       # 아미노산 치환 패턴 피처 on/off (CV로 효과 확인)
    USE_PATHWAYS=True,      # 경로 집계 피처 on/off
    USE_HOTSPOTS=True,      # gene|position, gene|trunc 토큰 on/off
    BALANCE=True,           # 클래스 불균형 보정 (sample weight / class_weight)
    XGB_N_EST=int(os.environ.get("XGB_N_EST", 2000)),
    LR_C=0.1,
    USE_COMBOS=True,        # 동반 변이 콤보 피처 on/off
    TMB_LOW=5,              # 극저변이 기준 (THCA/LAML 등)
    TMB_HIGH=30,            # 고변이 기준 (SKCM/UCEC/STES 등)
    COAD_TEST_PRIOR=0.22,   # 가이드가 말하는 test COAD 비율 (검증 시뮬레이션용)
    BOOST_MIN_GAIN=0.003,   # COAD 가중으로 시뮬 macroF1이 이만큼 안 오르면 적용 안 함
    RUN_ABLATION=False,     # True면 콤보 ON/OFF CV 비교 (시간 2배)
)


## 1. 데이터 로드

In [ ]:
D = CFG["DATA_DIR"]
train = pd.read_csv(f"{D}/train.csv").fillna("WT")
test = pd.read_csv(f"{D}/test.csv").fillna("WT")
sub = pd.read_csv(f"{D}/sample_submission.csv")

assert (test["ID"].values == sub["ID"].values).all(), "test와 sample_submission의 ID 순서가 다릅니다"
gene_cols = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
assert gene_cols == [c for c in test.columns if c != "ID"], "train/test 유전자 컬럼이 다릅니다"

le = LabelEncoder()
y = le.fit_transform(train["SUBCLASS"])
n_class = len(le.classes_)
print("train", train.shape, "test", test.shape, "classes", n_class)
print(train["SUBCLASS"].value_counts())

In [ ]:
train.head(5)

## EDA

In [ ]:
import pandas as pd
import numpy as np
import re

# 1. 데이터 로드
train = pd.read_csv(f"{D}/train.csv").fillna("WT")
test = pd.read_csv(f"{D}/test.csv").fillna("WT")

print("="*50)
print("1. 기본 데이터 크기 및 암종(SUBCLASS) 개수 확인")
print("="*50)
print(f"Train 크기: {train.shape}")
print(f"Test 크기: {test.shape}")

# 암종(SUBCLASS) 개수 및 분포 확인
subclass_counts = train["SUBCLASS"].value_counts()
print(f"총 암종(Subclass) 개수: {len(subclass_counts)}개")
print("\n[상위 5개 암종 분포]")
print(subclass_counts.head())

# 유전자 컬럼 분리
gene_cols = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
print(f"\n유전자(Feature) 컬럼 총 개수: {len(gene_cols)}개")


print("\n" + "="*50)
print("2. 샘플당 변이 유전자 수 분포 (TMB)")
print("="*50)
# WT가 아닌 유전자의 개수를 행별로 카운트
mut_matrix = (train[gene_cols] != "WT")
mut_counts_per_sample = mut_matrix.sum(axis=1)
print(mut_counts_per_sample.describe().round(2))


print("\n" + "="*50)
print("3. 한 셀에 변이가 여러 개 들어있는지 확인 (구분자 체크)")
print("="*50)
# WT가 아닌 셀들만 추출
all_vals = train[gene_cols].values.ravel()
non_wt_vals = all_vals[all_vals != "WT"]

non_wt_series = pd.Series(non_wt_vals)
# 쉼표(,), 콜론(:), 세미콜론(;), 파이프(|), 공백 등이 포함된 비율 확인
sep_pattern = r"[,:;|\s]"
has_sep = non_wt_series.str.contains(sep_pattern, regex=True).mean()
print(f"Non-WT 값 중 구분자(, : ; | 공백)가 포함된 비율: {has_sep:.4%}")


print("\n" + "="*50)
print("4. 변이 표기 패턴 종류 확인 (정규식 기반 상위 패턴)")
print("="*50)
# 숫자를 'N'으로 치환하여 대표적인 변이 표기 패턴 분석
pattern_samples = non_wt_series.str.replace(r"\d+", "N", regex=True)
print(pattern_samples.value_counts().head(15))


print("\n" + "="*50)
print("5. 유전자별 변이 빈도 (희소성 확인)")
print("="*50)
gene_mut_freq = mut_matrix.sum(axis=0)
print(f"변이가 단 1개라도 발생한 유전자 수: {(gene_mut_freq > 0).sum()} / {len(gene_cols)}")
print(f"대다수 유전자의 변이 빈도 요약 (상위/하위):")
print(gene_mut_freq.describe().round(2))

## 2. 변이 표기 형식 확인 (파싱 규칙이 맞는지 눈으로 검증)

In [ ]:
sample = pd.Series(train[gene_cols].iloc[:1000].to_numpy(dtype=object).ravel())
nonwt = sample[sample != "WT"]
print(f"non-WT 비율: {len(nonwt)/len(sample):.4%}")
print("한 셀에 구분자(, : ; | 공백)가 있는 비율:", nonwt.str.contains(r"[,:;|\s]").mean().round(4))
print(nonwt.str.replace(r"\d+", "N", regex=True).value_counts().head(25))

## 3. 피처 생성

- `g_*` : 유전자별 변이 여부 (이진)
- TMB 계열 : 변이 유전자 수, log, 이벤트 수
- 변이 유형 : missense / silent / nonsense / frameshift / splice / inframe 개수와 비율
- `hs_*` : 핫스팟(`유전자|원래아미노산+위치`) 및 truncating(`유전자|trunc`) 이진
- `aa_*` : 아미노산 치환쌍 비율
- `pw_*` : 경로별 변이 유전자 수 (아래 `PATHWAYS` 사전은 초안이니 자유롭게 수정)

In [ ]:
def combo_features(ex):
    mut = ex["mut"]; n = mut.shape[0]
    n_genes = mut.sum(1)
    low, high = n_genes <= CFG["TMB_LOW"], n_genes >= CFG["TMB_HIGH"]

    def G(name):   # 유전자 변이 여부 (없는 유전자는 0 벡터 — 엉뚱한 컬럼 안 가져옴)
        j = gene_pos.get(name)
        return mut[:, j].copy() if j is not None else np.zeros(n, bool)

    tok_rows = {}
    for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
        tok_rows.setdefault(nm, []).append(r)
    def T(name):   # 핫스팟/절단형 토큰 보유 여부 (MIN_HOTSPOT_FREQ와 무관하게 직접 계산)
        v = np.zeros(n, bool)
        if name in tok_rows: v[tok_rows[name]] = True
        return v

    f = {}
    # COAD
    f["COAD_APCtr_TP53"]  = T("APC|trunc") & G("TP53")
    f["COAD_APCtr_KRAS"]  = T("APC|trunc") & (G("KRAS") | G("FBXW7"))
    f["COAD_score"]       = T("APC|trunc").astype(int) + G("TP53") + G("KRAS") + G("FBXW7") + G("PIK3CA")
    # KIRC / KIPAN
    f["KIRC_VHL_PBRM1"]   = G("VHL") & G("PBRM1")
    f["KIRC_3p_score"]    = G("VHL").astype(int) + G("PBRM1") + G("SETD2") + G("BAP1")
    # UCEC
    f["UCEC_PTEN_CTNNB1"] = G("PTEN") & G("CTNNB1")
    f["UCEC_PTEN_PIK3CA"] = G("PTEN") & G("PIK3CA")
    f["UCEC_highTMB_PTEN"]= high & G("PTEN")
    # PAAD
    f["PAAD_score"]       = T("KRAS|G12").astype(int) + G("TP53") + T("CDKN2A|trunc") + G("SMAD4")
    # LGG vs GBMLGG
    f["LGG_IDH1_TP53_ATRX"] = T("IDH1|R132") & G("TP53") & G("ATRX")
    f["LGG_IDH1_clean"]   = T("IDH1|R132") & ~(G("PTEN") | G("EGFR"))
    f["GBM_PTEN_EGFR"]    = G("PTEN") & G("EGFR")
    f["GBM_noIDH1_signal"]= (G("PTEN") | G("EGFR")) & ~G("IDH1")
    # BRAF V600 + TMB (THCA vs SKCM vs COAD)
    f["THCA_BRAF_lowTMB"] = T("BRAF|V600") & low
    f["SKCM_BRAF_highTMB"]= T("BRAF|V600") & high
    f["SKCM_NRAS_highTMB"]= T("NRAS|Q61") & high
    f["COAD_BRAF_APC"]    = T("BRAF|V600") & T("APC|trunc")
    # 폐암 / 위식도 / 간 / 난소 / 백혈병
    f["LUAD_EGFR_noTP53"] = G("EGFR") & ~G("TP53")
    f["LUSC_TP53_noEGFR_highTMB"] = G("TP53") & ~G("EGFR") & high
    f["STES_highTMB_TP53_noAPC"]  = high & G("TP53") & ~T("APC|trunc")
    f["LIHC_CTNNB1_noAPC"]= G("CTNNB1") & ~G("APC")
    f["OV_TP53_lowTMB"]   = G("TP53") & (n_genes <= 20)
    f["LAML_NPM1_lowTMB"] = G("NPM1") & low

    names = list(f.keys())
    return np.c_[[np.asarray(f[k], np.float32) for k in names]].T, names

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"
AA_IDX = {a: i for i, a in enumerate(AA)}
TYPES = ["missense", "silent", "nonsense", "frameshift", "splice", "inframe", "other"]
TYPE_IDX = {t: i for i, t in enumerate(TYPES)}
SEP = re.compile(r"[,:;|\s]+")
RE_MIS = re.compile(r"^(?:p\.)?([A-Z])(\d+)([A-Z])$")
RE_NON = re.compile(r"^(?:p\.)?[A-Z]\d+(\*|X|Ter)$")

def parse_token(tok):
    """변이 표기 1개 -> (유형, 원래aa, 위치, 바뀐aa). 표기가 다르면 여기 규칙을 수정하세요."""
    low = tok.lower()
    m = re.search(r"\d+", tok)
    pos = m.group() if m else ""
    if "fs" in low or "frameshift" in low:
        return "frameshift", "", pos, ""
    if "splice" in low:
        return "splice", "", pos, ""
    if "del" in low or "ins" in low or "dup" in low:
        return "inframe", "", pos, ""
    if RE_NON.match(tok):
        return "nonsense", "", pos, ""
    m = RE_MIS.match(tok)
    if m:
        fa, p, ta = m.groups()
        return ("silent" if fa == ta else "missense"), fa, p, ta
    return "other", "", pos, ""

def extract(df):
    arr = df[gene_cols].to_numpy(dtype=object)
    mut = arr != "WT"
    n = len(df)
    type_cnt = np.zeros((n, len(TYPES)), np.float32)
    aa_cnt = np.zeros((n, 400), np.float32)
    tok_rows, tok_names = [], []
    rr, cc = np.nonzero(mut)
    for r, c in zip(rr, cc):
        gene = gene_cols[c]
        for tok in SEP.split(str(arr[r, c])):
            if not tok or tok == "WT":
                continue
            typ, fa, pos, ta = parse_token(tok)
            type_cnt[r, TYPE_IDX[typ]] += 1
            if typ == "missense":
                tok_rows.append(r); tok_names.append(f"{gene}|{fa}{pos}")
                if fa in AA_IDX and ta in AA_IDX:
                    aa_cnt[r, AA_IDX[fa] * 20 + AA_IDX[ta]] += 1
            elif typ in ("nonsense", "frameshift", "splice"):
                tok_rows.append(r); tok_names.append(f"{gene}|trunc")
            elif typ == "inframe":                       # ← 추가
                tok_rows.append(r); tok_names.append(f"{gene}|inframe")        # 유전자 단위
                tok_rows.append(r); tok_names.append(f"{gene}|{pos}_inframe")  # 위치 단위 (EGFR 746_inframe 등)
    return dict(mut=mut, type_cnt=type_cnt, aa_cnt=aa_cnt,
                tok_rows=np.array(tok_rows, dtype=np.int64), tok_names=tok_names)

ex_tr, ex_te = extract(train), extract(test)
print("파싱 완료. 유형별 총 개수(train):", dict(zip(TYPES, ex_tr["type_cnt"].sum(0).astype(int))))

In [ ]:
# ---- 경로/유전자군 사전 (초안: 필요에 맞게 수정/추가) ----
PATHWAYS = {
    "p53": ["TP53","MDM2","MDM4","CDKN2A","ATM","CHEK1","CHEK2"],
    "RTK_RAS": ["EGFR","ERBB2","ERBB3","ERBB4","FGFR1","FGFR2","FGFR3","FGFR4","MET","ALK","RET","KIT",
                "PDGFRA","KRAS","NRAS","HRAS","BRAF","RAF1","NF1","MAP2K1","MAP2K2","SOS1","PTPN11"],
    "PI3K": ["PIK3CA","PIK3CB","PIK3R1","PTEN","AKT1","AKT2","AKT3","MTOR","TSC1","TSC2","STK11","RICTOR","INPP4B"],
    "Wnt": ["APC","CTNNB1","AXIN1","AXIN2","RNF43","ZNRF3","TCF7L2","GSK3B","AMER1"],
    "CellCycle": ["RB1","CDK4","CDK6","CCND1","CCND2","CCND3","CCNE1","CDKN1A","CDKN1B","CDKN2B","E2F1","FBXW7"],
    "Notch": ["NOTCH1","NOTCH2","NOTCH3","NOTCH4","FBXW7","SPEN"],
    "Hippo": ["NF2","LATS1","LATS2","SAV1","YAP1","WWTR1","TEAD1"],
    "TGFb": ["TGFBR1","TGFBR2","SMAD2","SMAD3","SMAD4","ACVR1B","ACVR2A"],
    "NRF2": ["NFE2L2","KEAP1","CUL3"],
    "SWI_SNF": ["ARID1A","ARID1B","ARID2","SMARCA4","SMARCB1","PBRM1","SMARCA2"],
    "Chromatin": ["KMT2A","KMT2B","KMT2C","KMT2D","EZH2","SETD2","KDM6A","KDM5C","CREBBP","EP300",
                  "BAP1","ASXL1","DNMT3A","IDH1","IDH2","ATRX"],
    "DDR_MMR": ["BRCA1","BRCA2","PALB2","ATR","POLE","POLD1","MLH1","MSH2","MSH6","PMS2",
                "RAD51","RAD50","MRE11","FANCA","FANCD2","ERCC2"],
    "Splicing": ["SF3B1","SRSF2","U2AF1","RBM10"],
    "LongGenes": ["TTN","MUC16","MUC4","SYNE1","OBSCN","RYR2","RYR1","CSMD1","CSMD3","LRP1B","USH2A","FLG","PCLO"],
}
gene_pos = {g: i for i, g in enumerate(gene_cols)}
PW_IDX = {k: [gene_pos[g] for g in v if g in gene_pos] for k, v in PATHWAYS.items()}
print({k: f"{len(v)}/{len(PATHWAYS[k])}" for k, v in PW_IDX.items()})   # 데이터에 존재하는 유전자 수/사전 크기

In [ ]:
# ---- train 기준으로만 사용할 유전자/토큰/아미노산쌍 결정 (test는 보지 않음: Data Leakage 방지) ----
# test는 아래 compose()에서 이 기준에 맞춰 변환만 한다. test에만 있는 토큰은 tok_idx.get()이 None이라 무시됨.
freq = ex_tr["mut"].sum(0)
keep_idx = np.where(freq >= CFG["MIN_GENE_FREQ"])[0]

def uniq_pairs(ex):
    return set(zip(ex["tok_rows"].tolist(), ex["tok_names"]))

tok_cnt = Counter(nm for _, nm in uniq_pairs(ex_tr))
keep_tokens = sorted(t for t, c in tok_cnt.items() if c >= CFG["MIN_HOTSPOT_FREQ"])
tok_idx = {t: i for i, t in enumerate(keep_tokens)}

aa_tot = ex_tr["aa_cnt"].sum(0)
aa_keep = np.where(aa_tot >= CFG["MIN_AA_FREQ"])[0]
print(f"유전자 {len(keep_idx)}/{len(gene_cols)}, 핫스팟 토큰 {len(keep_tokens)}, 아미노산쌍 {len(aa_keep)}")

def compose(ex):
    mut = ex["mut"]; n = mut.shape[0]
    blocks, names = [], []
    def add(a, nm):
        blocks.append(np.asarray(a, np.float32).reshape(n, -1)); names.extend(nm)

    add(mut[:, keep_idx], [f"g_{gene_cols[i]}" for i in keep_idx])

    n_genes = mut.sum(1)
    tc = ex["type_cnt"]; n_ev = tc.sum(1)
    add(np.c_[n_genes, np.log1p(n_genes), n_ev], ["n_mut_genes", "log_n_mut_genes", "n_events"])
    add(tc, [f"cnt_{t}" for t in TYPES])
    trunc = tc[:, [TYPE_IDX[t] for t in ("nonsense", "frameshift", "splice")]].sum(1)
    add(np.c_[tc[:, TYPE_IDX["silent"]] / (n_ev + 1),
              tc[:, TYPE_IDX["missense"]] / (n_ev + 1),
              trunc / (n_ev + 1)], ["ratio_silent", "ratio_missense", "ratio_trunc"])

    if CFG["USE_HOTSPOTS"] and keep_tokens:
        H = np.zeros((n, len(keep_tokens)), np.float32)
        for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
            j = tok_idx.get(nm)
            if j is not None:
                H[r, j] = 1
        add(H, [f"hs_{t}" for t in keep_tokens])
    if CFG["USE_COMBOS"]:
        C, cn = combo_features(ex)
        add(C, [f"cb_{k}" for k in cn])
    if CFG["USE_AA_SUBS"] and len(aa_keep):
        A = ex["aa_cnt"][:, aa_keep] / (tc[:, TYPE_IDX["missense"]] + 1)[:, None]
        add(A, [f"aa_{AA[k // 20]}>{AA[k % 20]}" for k in aa_keep])

    if CFG["USE_PATHWAYS"]:
        P = np.c_[[mut[:, idx].sum(1) if idx else np.zeros(n) for idx in PW_IDX.values()]].T
        add(P, [f"pw_{k}" for k in PW_IDX])

    return np.hstack(blocks), names

X, feat_names = compose(ex_tr)
X_te, _ = compose(ex_te)
print("X", X.shape, "X_te", X_te.shape)

## 4. 5-fold CV (XGBoost + Logistic Regression)

※ XGB는 검증 fold로 early stopping을 하므로 OOF 점수가 아주 조금 낙관적일 수 있습니다 (모델 간 비교용으로는 충분).


In [ ]:
def run_cv(X, X_te, verbose=True):
    skf = StratifiedKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
    oof_xgb = np.zeros((len(y), n_class)); te_xgb = np.zeros((len(X_te), n_class))
    oof_lr = np.zeros((len(y), n_class));  te_lr = np.zeros((len(X_te), n_class))
    imp = np.zeros(X.shape[1])

    for fold, (tr, va) in enumerate(skf.split(X, y)):
        assert len(np.unique(y[tr])) == n_class, "일부 클래스가 train fold에 없습니다 (N_FOLDS를 줄이세요)"
        sw = compute_sample_weight("balanced", y[tr]) if CFG["BALANCE"] else None

        m = xgb.XGBClassifier(
            objective="multi:softprob", n_estimators=CFG["XGB_N_EST"], learning_rate=0.08,
            max_depth=5, subsample=0.8, colsample_bytree=0.3, tree_method="hist",
            eval_metric="mlogloss", early_stopping_rounds=50, random_state=CFG["SEED"], n_jobs=-1,
        )
        m.fit(X[tr], y[tr], sample_weight=sw, eval_set=[(X[va], y[va])], verbose=False)
        oof_xgb[va] = m.predict_proba(X[va]); te_xgb += m.predict_proba(X_te) / CFG["N_FOLDS"]
        imp += m.feature_importances_ / CFG["N_FOLDS"]

        lr = make_pipeline(MaxAbsScaler(), LogisticRegression(
            C=CFG["LR_C"], max_iter=500, class_weight="balanced" if CFG["BALANCE"] else None))
        lr.fit(X[tr], y[tr])
        oof_lr[va] = lr.predict_proba(X[va]); te_lr += lr.predict_proba(X_te) / CFG["N_FOLDS"]

        if verbose:
            f = lambda p: f1_score(y[va], p[va].argmax(1), average="macro")
            print(f"fold{fold} | XGB(best_iter={m.best_iteration}) macroF1={f(oof_xgb):.4f} | LR macroF1={f(oof_lr):.4f}")
    return oof_xgb, te_xgb, oof_lr, te_lr, imp

oof_xgb, te_xgb, oof_lr, te_lr, imp = run_cv(X, X_te)


In [ ]:
# ---- (선택) 콤보 피처 효과 확인: CFG["RUN_ABLATION"]=True로 켜서 실행 ----
if CFG["RUN_ABLATION"]:
    keep_flag = CFG["USE_COMBOS"]
    CFG["USE_COMBOS"] = False
    X0, _ = compose(ex_tr); X0_te, _ = compose(ex_te)
    CFG["USE_COMBOS"] = keep_flag
    o_x, _, o_l, _, _ = run_cv(X0, X0_te, verbose=False)
    for nm, p in [("콤보 OFF  XGB", o_x), ("콤보 OFF  LR ", o_l), ("콤보 ON   XGB", oof_xgb), ("콤보 ON   LR ", oof_lr)]:
        print(f"{nm}  macroF1={f1_score(y, p.argmax(1), average='macro'):.4f}  acc={accuracy_score(y, p.argmax(1)):.4f}")
else:
    print("ablation 건너뜀 (CFG['RUN_ABLATION']=True로 실행 가능)")


## 5. 평가 · 앙상블 · 오분류 분석

In [ ]:
def report(name, p):
    print(f"{name:12s} acc={accuracy_score(y, p.argmax(1)):.4f}  macroF1={f1_score(y, p.argmax(1), average='macro'):.4f}")

report("XGB", oof_xgb); report("LR", oof_lr)

best_w, best_s = 0.5, -1
for w in np.linspace(0, 1, 11):
    s = f1_score(y, (w * oof_xgb + (1 - w) * oof_lr).argmax(1), average="macro")
    if s > best_s: best_w, best_s = w, s
oof_ens = best_w * oof_xgb + (1 - best_w) * oof_lr
te_ens = best_w * te_xgb + (1 - best_w) * te_lr
print(f"\n앙상블 XGB 가중치={best_w:.1f}")
report("ENSEMBLE", oof_ens)

print("\n" + classification_report(y, oof_ens.argmax(1), target_names=le.classes_, digits=3))

cm = confusion_matrix(y, oof_ens.argmax(1)); np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], le.classes_[i], le.classes_[j]) for i in range(n_class) for j in range(n_class) if cm[i, j] > 0), reverse=True)[:10]
print("가장 많이 헷갈리는 쌍 (정답 -> 예측):")
for c, a, b in pairs: print(f"  {a} -> {b}: {c}")

top = np.argsort(imp)[::-1][:20]
print("\nXGB 중요도 상위 20:", [feat_names[i] for i in top])

In [ ]:
# ---- COAD 확률 가중 (선택, 자동 판단) ----
# 전체 클래스 prior 보정(EM)은 소수 클래스를 깎아 macroF1을 망가뜨림(0.46 -> 0.21).
# 그래서 COAD 한 클래스만 k배 키우는 방식을 'COAD 22% 분포로 재표집한 OOF'에서 검증하고,
# 이득이 BOOST_MIN_GAIN 미만이면 k=1(미적용)로 둔다.
te_final = te_ens.copy()
if "COAD" in le.classes_:
    coad = int(le.transform(["COAD"])[0])
    def boost(P, k):
        Q = P.copy(); Q[:, coad] *= k
        return Q / Q.sum(1, keepdims=True)

    tr_prior = np.bincount(y) / len(y)
    tgt = tr_prior * ((1 - CFG["COAD_TEST_PRIOR"]) / (1 - tr_prior[coad])); tgt[coad] = CFG["COAD_TEST_PRIOR"]
    rng = np.random.default_rng(CFG["SEED"]); N = 6000
    pools = [np.where(y == c)[0] for c in range(n_class)]
    draws = [np.concatenate([rng.choice(pools[c], max(1, int(round(tgt[c] * N))), replace=True) for c in range(n_class)])
             for _ in range(20)]

    grid = [1.0, 1.25, 1.5, 2.0, 3.0]
    res = {k: float(np.mean([f1_score(y[ix], boost(oof_ens[ix], k).argmax(1), average="macro") for ix in draws])) for k in grid}
    print("COAD 22% 가정 시뮬 macroF1:", {k: round(v, 4) for k, v in res.items()})

    best_k = max(res, key=res.get)
    if res[best_k] - res[1.0] < CFG["BOOST_MIN_GAIN"]:
        best_k = 1.0
    te_final = boost(te_ens, best_k)
    print(f"선택된 k={best_k}  | test COAD 예측 수: {int((te_ens.argmax(1)==coad).sum())} -> {int((te_final.argmax(1)==coad).sum())}"
          f"  (가이드 기준 약 {int(CFG['COAD_TEST_PRIOR']*len(te_ens))}명)")


## 6. 제출 파일 생성

In [ ]:
pred = le.inverse_transform(te_final.argmax(1))
out = sub.copy()
out["SUBCLASS"] = pred
assert len(out) == len(test) and set(out["SUBCLASS"]) <= set(le.classes_)
out.to_csv(f"{D}/submission_v3.csv", index=False, encoding="UTF-8-sig")
np.save(f"{D}/oof_ens_v3.npy", oof_ens); np.save(f"{D}/te_final_v3.npy", te_final)   # 이후 앙상블/후처리용
print(out["SUBCLASS"].value_counts().head(10))
